# Atulya remote brain on Colab

Run a **bigger model on Colab's free GPU** and let Atulya on your PC use it. About 5 minutes the first time.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run the cells in order. At the end you get an **address and a key** to give Atulya.

**Please read**
- Anyone with the address *and* the key can use your Colab GPU. Keep the key private. Both change every time you run this.
- Your questions travel through Cloudflare and Google. It is **not private**. Atulya's AI check for unsure bank messages also sends (redacted) text to whichever brain answers; set `ATULYA_MONEY_AI=off` in `.env` if that matters to you.
- Free Colab ends sessions after a while and has its own usage rules. If it disconnects, run the notebook again for a new address.
- This notebook was written from the documentation and has not been run on Colab by its author. If a cell fails, the message will say which step.

In [ ]:
# 1. Is there a GPU?
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(out or "NO GPU FOUND: Runtime > Change runtime type > T4 GPU, then run again.")

In [ ]:
# 2. Install the model server with GPU support (first try the ready-made build, about 1 minute)
%%bash
pip install -q huggingface_hub requests "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 2>&1 | tail -3

In [ ]:
# 3. Did it get GPU support? If this prints False, run the fallback cell below (about 10 minutes).
from llama_cpp import llama_supports_gpu_offload
print("GPU offload available:", llama_supports_gpu_offload())

In [ ]:
# 3b. Fallback only if step 3 printed False: build it for this machine's GPU (takes ~10 minutes, then restart the runtime)
# %%bash
# CMAKE_ARGS="-DGGML_CUDA=on" pip install --force-reinstall --no-cache-dir "llama-cpp-python[server]"

In [ ]:
# 4. Pick and download a model (public Qwen3 files; a T4 has 16 GB of memory)
MODEL = "8b"  #@param ["4b", "8b", "14b"]
FILES = {"4b": ("unsloth/Qwen3-4B-GGUF", "Qwen3-4B-Q4_K_M.gguf"),
         "8b": ("unsloth/Qwen3-8B-GGUF", "Qwen3-8B-Q4_K_M.gguf"),
         "14b": ("unsloth/Qwen3-14B-GGUF", "Qwen3-14B-Q4_K_M.gguf")}
from huggingface_hub import hf_hub_download
repo, file = FILES[MODEL]
path = hf_hub_download(repo, file, local_dir="/content/models")
ALIAS = f"qwen3-{MODEL}"
print("ready:", path)

In [ ]:
# 5. Start the model server, protected by a random key
import os, secrets, subprocess, time, requests
API_KEY = secrets.token_urlsafe(24)
log = open("/content/server.log", "w")
server = subprocess.Popen(
    ["python", "-m", "llama_cpp.server", "--model", path, "--model_alias", ALIAS, "--n_gpu_layers", "-1",
     "--n_ctx", "8192", "--host", "127.0.0.1", "--port", "8000", "--api_key", API_KEY],
    stdout=log, stderr=subprocess.STDOUT)
for _ in range(150):
    try:
        if requests.get("http://127.0.0.1:8000/v1/models", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=2).ok:
            break
    except requests.RequestException:
        pass
    if server.poll() is not None:
        raise RuntimeError("The model server stopped. Last lines of /content/server.log:\n" + open("/content/server.log").read()[-1500:])
    time.sleep(2)
else:
    raise RuntimeError("The model server did not start in time. See /content/server.log")
print("model server is running")

In [ ]:
# 6. Open a public https address (Cloudflare quick tunnel, no account needed)
import re, urllib.request
urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared")
os.chmod("/content/cloudflared", 0o755)
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for _ in range(120):
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0)
        break
if not URL:
    raise RuntimeError("Could not open the tunnel. Run this cell again.")
print("address ready")

In [ ]:
# 7. Test it through the public address, and measure the speed
t = time.time()
r = requests.post(URL + "/v1/chat/completions", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=120,
                  json={"model": ALIAS, "max_tokens": 120, "messages": [{"role": "user", "content": "In two sentences: why does a monthly budget help? /no_think"}]})
r.raise_for_status()
dt = time.time() - t
n = r.json()["usage"]["completion_tokens"]
print(r.json()["choices"][0]["message"]["content"].strip())
print(f"\n{n} tokens in {dt:.1f}s = {n / dt:.1f} tokens/s")
print("\n" + "=" * 60)
print("ON YOUR PC, in the Atulya folder, run:\n")
print(f"  python examples/connect_remote.py --url {URL}/v1 --key {API_KEY} --model {ALIAS}\n")
print("(or paste the same three values in Menu > Brains & keys > 'Your own'). Then restart start.bat.")
print("=" * 60)

In [ ]:
# 8. Keep this running while you use Atulya. Stop this cell (or close the tab) to end the link.
try:
    while True:
        time.sleep(60)
        print(".", end="", flush=True)
except KeyboardInterrupt:
    tunnel.terminate(); server.terminate()
    print("\nStopped.")